# Who picks the tool? A walkthrough

This notebook recomputes every number in the MeasureCamp talk from the committed runs, and shows you how to
make new calls if you have keys. It runs without any API key.

**The question.** An AI support assistant gets a customer message and 91 tools (90 copied verbatim from
official MCP servers, plus `hand_off_to_human`). Which tool should it call first? We compare four selectors:

- **Jev** (TypeSafe System One): a small model that answers a `choice` question with a probability per option.
- **DeepSeek V4.1 Flash, Claude Haiku 5.5, Claude Sonnet 5.5**, each in two modes: as a *router* (told to
  reply with one tool's name) and with *native tool calling* (the tools in the API's `tools` parameter).

The **final setup**, used for the talk's main results, runs on 600 *fresh* messages: every selector gets the same
routing policy, six tool descriptions get one line in customers' words, and the LLMs work as real agents (native
tool calling). The **development runs** on the first 600 messages show how we got there: plain and policy prompts,
the router mode, and a first-pass answer key that we later audited. The ranking depends on those choices, so this
notebook lets you see each one.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import toolpick as tp

pd.set_option("display.max_colwidth", 90)
print(len(tp.requests("fresh")), "fresh messages ·", len(tp.tools()), "tools")
print("final runs:", tp.available_runs("fresh"))
print("development runs:", tp.available_runs())

600 fresh messages · 91 tools
final runs: ['fresh/jev', 'fresh/native_deepseek', 'fresh/native_haiku', 'fresh/native_sonnet']
development runs: ['jev', 'jev_policy', 'native_deepseek', 'native_deepseek_policy', 'native_haiku', 'native_haiku_policy', 'native_sonnet', 'native_sonnet_policy', 'router_deepseek', 'router_deepseek_gepa', 'router_deepseek_policy', 'router_haiku', 'router_haiku_policy', 'router_sonnet', 'router_sonnet_policy']


## 1. The data

Two samples of 600 from Bitext's customer-support dataset. The text is template-generated with deliberate typos;
the `intent` is Bitext's label, which we map to tools. The fresh sample shares no message with the first.

In [2]:
from dataclasses import asdict

reqs = pd.DataFrame([asdict(r) for r in tp.requests("fresh")])
display(reqs.sample(8, random_state=1))
reqs["intent"].value_counts().to_frame("messages").T

,id,intent,text
446,F0446,delivery_period,"I am looking for shipping periods, where could I find them?"
404,F0404,registration_problems,can you help me informing of a registration error?
509,F0509,delivery_period,could i check when my package is gonna arive
455,F0455,complaint,can ya help me lodging an consumer complaint
201,F0201,place_order,can uhelp me buying an artikcle
23,F0023,contact_human_agent,need help talking to somebody
415,F0415,change_shipping_address,"my address has changed, could you help me changing it?"
548,F0548,change_order,update purchase00004587345


intent,change_order,check_cancellation_fee,check_payment_methods,change_shipping_address,cancel_order,check_invoice,complaint,recover_password,create_account,delivery_options,...,contact_human_agent,edit_account,payment_issue,track_order,registration_problems,set_up_shipping_address,delete_account,get_refund,review,check_refund_policy
messages,23,23,23,23,23,23,22,22,22,22,...,22,22,22,22,22,22,22,22,22,22


The tools. Every name, description and schema was taken from the vendor: by running the official package and
calling `tools/list`, by calling a live public Shopify store, or from official docs (Intercom). `data/REPORT.md`
has the provenance of each server. Note how uneven the prompt budget is.

In [3]:
cat = pd.DataFrame(tp.catalog())[["qualified_name", "server", "description", "exactness"]]
display(cat.groupby("server").size().to_frame("tools").T)
tok = tp.tool_tokens_by_server()
print({k: f"{v / sum(tok.values()):.0%}" for k, v in sorted(tok.items(), key=lambda kv: -kv[1])}, "of the tool JSON")
cat[cat["server"] == "shopify_customer_account"]

server,github,intercom,notion,shopify_customer_account,shopify_storefront,shopify_ucp,stripe
tools,10,14,24,4,1,15,22


{'notion': '41%', 'shopify_ucp': '38%', 'stripe': '10%', 'github': '8%', 'intercom': '1%', 'shopify_customer_account': '1%', 'shopify_storefront': '1%'} of the tool JSON


,qualified_name,server,description,exactness
16,shopify_customer_account.get_most_recent_order_status,shopify_customer_account,Gets the status of the customer's most recent order.,exact (tools/list)
17,shopify_customer_account.get_order_status,shopify_customer_account,Fetches a specific order by order number for a logged-in customer and returns status d...,exact (tools/list)
18,shopify_customer_account.get_store_credit_balances,shopify_customer_account,Fetches the customer's store credit balances if they have any.,exact (tools/list)
19,shopify_customer_account.request_return,shopify_customer_account,Allows a customer to request a return for an order.\nEither an order_id or an order_nu...,exact (tools/list)


## 2. The labels

Per intent: the `acceptable` tools (correct) and the `partial` tools (a benign first step that doesn't finish the
job). **14 of the 27 intents have no real tool**; there the correct answer is `hand_off_to_human`. `ask_ok` marks
the intents where asking the customer a question is correct, because every correct tool needs details the message
doesn't give.

The default answer key is `audited`: every failure in the development runs was checked by hand, and each change has
its reason (the `changes` column). `first_pass` is what the study started with. Both live in `src/toolpick/data.py`.

In [4]:
lab = tp.labels()
pd.DataFrame([{"intent": k, "no tool": v["gap"], "acceptable": ", ".join(t.split(".")[-1] for t in v["acceptable"]),
               "partial": ", ".join(t.split(".")[-1] for t in v["partial"]), "ask_ok": v["ask_ok"], "changes": " ".join(v["changes"])}
              for k, v in lab.items()]).sort_values(["no tool", "intent"])

,intent,no tool,acceptable,partial,ask_ok,changes
3,check_cancellation_fee,False,search_shop_policies_and_faqs,"get_checkout, get_cart, get_order, search_catalog, lookup_catalog, get_product, search...",False,
4,check_invoice,False,"list_invoices, get_order_status, get_most_recent_order_status, get_order, list_customers","search_shop_policies_and_faqs, get_checkout, get_cart, search_catalog, lookup_catalog,...",False,For a shopper the bill is usually the order receipt; Stripe needs the customer before ...
5,check_payment_methods,False,search_shop_policies_and_faqs,"get_checkout, create_checkout, get_cart, get_order, search_catalog, lookup_catalog, ge...",False,
6,check_refund_policy,False,search_shop_policies_and_faqs,"get_checkout, get_cart, get_order, search_catalog, lookup_catalog, get_product, search...",False,
8,contact_customer_service,False,"search_shop_policies_and_faqs, hand_off_to_human","get_checkout, get_cart, get_order, search_catalog, lookup_catalog, get_product, search...",False,Handing to a person is a defensible reading of 'contact customer service'. 'I want to ...
12,delivery_options,False,search_shop_policies_and_faqs,"search_locations, create_checkout, get_checkout, get_cart, get_order, search_catalog, ...",False,
13,delivery_period,False,"search_shop_policies_and_faqs, get_order_status, get_most_recent_order_status, get_order","get_checkout, get_cart, search_catalog, lookup_catalog, get_product, search_locations,...",False,"get_order is accepted for track_order, so it is accepted here too."
15,get_invoice,False,"list_invoices, get_order_status, get_most_recent_order_status, get_order, list_customers","search_shop_policies_and_faqs, get_checkout, get_cart, search_catalog, lookup_catalog,...",False,Same as check_invoice.
16,get_refund,False,"request_return, create_refund, get_order_status, get_most_recent_order_status, get_ord...","search_shop_policies_and_faqs, get_checkout, get_cart, search_catalog, lookup_catalog,...",True,"Both refund tools need an order or payment ID first, so the lookup is the right first ..."
18,payment_issue,False,"list_payment_intents, hand_off_to_human","list_disputes, search_shop_policies_and_faqs, get_checkout, get_cart, get_order, searc...",True,Someone reporting a payment problem should reach a person. Investigating needs to know...


## 3. Scoring one decision

Every row of a run has the selector's pick (or no pick, if a native model replied in text). `tp.outcome` turns a
pick into one of six outcomes, which fold into three verdicts: **correct** (`right`), **benign** (`defensible`,
`handed_off`, `asked`: the customer waits, nothing breaks) and **harmful** (`wrong`, `bluffed`: a tool that can't
do what was asked, so it looks as if something happened).

In [5]:
for pick in ["shopify_customer_account.get_order_status", "shopify_ucp.get_order", "hand_off_to_human",
             "shopify_storefront.search_shop_policies_and_faqs", None]:
    print(f"{str(pick):55s} track_order -> {tp.outcome(pick, lab['track_order']):12s}  cancel_order -> {tp.outcome(pick, lab['cancel_order'])}")

shopify_customer_account.get_order_status               track_order -> right         cancel_order -> defensible
shopify_ucp.get_order                                   track_order -> right         cancel_order -> defensible
hand_off_to_human                                       track_order -> handed_off    cancel_order -> right
shopify_storefront.search_shop_policies_and_faqs        track_order -> defensible    cancel_order -> defensible
None                                                    track_order -> asked         cancel_order -> asked


## 4. The headline table

`scored_rows` joins a run with the answer key; `summarise` gives the rates. First the final setup on the fresh
messages, then the development runs. Change `version` to `"first_pass"` and watch the ranking move.

In [6]:
def table(sample="fresh", version="audited"):
    rows = []
    for name in tp.available_runs(sample):
        s = tp.summarise(tp.scored_rows(name, version))
        rows.append({"run": name, "n": s["n"], "correct": s["correct"], "benign": s["benign"], "harmful": s["harmful"], "asked": s["asked"],
                     "handed_off": s["handoff"], "latency_s": s["latency_median_s"], "input_tokens": s["input_tokens_median"],
                     "$ per 1000": s["cost_per_1000"]})
    return pd.DataFrame(rows).set_index("run").sort_values("correct", ascending=False)


def show(t):
    return (t.style.format({"correct": "{:.1%}", "benign": "{:.1%}", "harmful": "{:.1%}", "asked": "{:.1%}", "handed_off": "{:.1%}",
                            "latency_s": "{:.2f}", "input_tokens": "{:,.0f}", "$ per 1000": "{:.2f}"})
             .bar(subset=["correct"], color="#8ab17d", vmin=0, vmax=1).bar(subset=["harmful"], color="#e9a3a3", vmin=0, vmax=1))


show(table("fresh"))

,n,correct,benign,harmful,asked,handed_off,latency_s,input_tokens,$ per 1000
run,,,,,,,,,
fresh/jev,600,95.3%,4.5%,0.2%,0.0%,61.0%,0.26,"7,207",0.30
fresh/native_sonnet,600,93.3%,6.7%,0.0%,4.5%,52.0%,1.79,"76,222",9.07
fresh/native_haiku,600,82.2%,17.8%,0.0%,13.0%,41.7%,1.65,"76,221",0.85
fresh/native_deepseek,600,47.2%,52.5%,0.3%,36.3%,7.5%,2.02,"51,609",0.28


In [7]:
pd.concat({"audited key": table("first")[["correct", "harmful"]], "first-pass key": table("first", "first_pass")[["correct", "harmful"]]},
          axis=1).style.format("{:.1%}")

## 5. Paired comparisons

Two accuracies on the same messages aren't independent, so we don't compare them as if they were. `paired`
counts the messages where only one of the two was right (an exact McNemar test) and bootstraps the difference.

In [8]:
pairs = []
todo = [("fresh/jev", f"fresh/native_{m}") for m in ("deepseek", "haiku", "sonnet")]
todo += [("jev" + s_, o + s_) for o in ("router_deepseek", "router_sonnet", "native_sonnet") for s_ in ("", "_policy")]
for a, other in todo:
        p = tp.paired(a, other)
        pairs.append({"jev vs": other, "n": p["n_pairs"], "jev": p["right_a"], "other": p["right_b"],
                      "only jev right": p["only_a"], "only other right": p["only_b"], "diff": p["diff"],
                      "95% CI": f"[{p['ci'][0]:+.3f}, {p['ci'][1]:+.3f}]", "p": p["p"]})
pd.DataFrame(pairs).style.format({"jev": "{:.1%}", "other": "{:.1%}", "diff": "{:+.1%}", "p": "{:.2g}"})

,jev vs,n,jev,other,only jev right,only other right,diff,95% CI,p
0,fresh/native_deepseek,600,95.3%,47.2%,308,19,+48.2%,"[+0.435, +0.528]",2.2e-68
1,fresh/native_haiku,600,95.3%,82.2%,102,23,+13.2%,"[+0.097, +0.168]",4.6e-13
2,fresh/native_sonnet,600,95.3%,93.3%,37,25,+2.0%,"[-0.007, +0.045]",0.16
3,router_deepseek,600,77.7%,73.5%,85,60,+4.2%,"[+0.002, +0.078]",0.046
4,router_deepseek_policy,600,85.5%,86.0%,41,44,-0.5%,"[-0.033, +0.025]",0.83
5,router_sonnet,300,77.3%,70.3%,56,35,+7.0%,"[+0.010, +0.133]",0.035
6,router_sonnet_policy,300,84.7%,92.7%,10,34,-8.0%,"[-0.123, -0.040]",0.00039
7,native_sonnet,600,77.7%,59.0%,203,91,+18.7%,"[+0.133, +0.240]",5.6e-11
8,native_sonnet_policy,600,85.5%,92.2%,22,62,-6.7%,"[-0.097, -0.037]",1.5e-05


## 6. Error analysis

Three views. First: when a selector is harmful, what does it reach for?

In [9]:
def wrong_calls(name):
    rows = pd.DataFrame(tp.scored_rows(name))
    w = rows[rows["verdict"] == "harmful"]
    return (w.assign(pick=w["pick"].str.split(".").str[-1]).groupby(["intent", "pick"]).size()
             .sort_values(ascending=False).head(10).to_frame("n"))

display(wrong_calls("fresh/native_haiku"))
wrong_calls("fresh/native_sonnet")

,,n
intent,pick,


,,n
intent,pick,


Second: Jev's errors and its confidence. When Jev is wrong it mostly hands off (benign). Its confidence tracks
whether it's right, which lets you send only the unsure messages to a bigger model.

In [10]:
jev = pd.DataFrame(tp.scored_rows("fresh/jev"))
display(jev.groupby("outcome").size().to_frame("messages"))
jev["band"] = pd.cut(jev["confidence"], [0, .7, .9, 1.01], right=False, labels=["below 0.7", "0.7-0.9", "0.9+"])
jev.groupby("band", observed=True)["right"].agg(["size", "mean"]).rename(columns={"size": "messages", "mean": "correct"})

,messages
outcome,
defensible,1
handed_off,26
right,572
wrong,1


,messages,correct
band,,
below 0.7,111,0.855856
0.7-0.9,132,0.931818
0.9+,357,0.991597


Third: the native models often reply in text instead of calling a tool. On vague messages that's reasonable; the
router condition can't do it. Scoring the first turn only makes native accuracy a lower bound.

In [11]:
nat = pd.DataFrame(tp.scored_rows("fresh/native_sonnet"))
asked = nat[nat["outcome"] == "asked"]
print(f"{len(asked)} of {len(nat)} replied in text")
asked[["intent", "text", "reply"]].sample(5, random_state=2)

27 of 600 replied in text


,intent,text,reply
538,contact_human_agent,contactingassistant,Hi! It looks like your message may have been cut off. What can I help you with today? ...
18,get_invoice,wanna get the bil from Chloe will uhelp me,I can help with a bill or invoice. I need a bit more information first:\n\n- Is this y...
229,recover_password,problem with forgotten keys,I'm sorry you're having trouble with forgotten keys. I need a bit more detail to help....
422,check_cancellation_fee,i have got to see trhe withdrawal charge can i get some help,I want to make sure I look in the right place. Which of these do you mean?\n\n1. **A c...
374,switch_account,use standard acdcount,"The message ""use standard acdcount"" is unclear to me, and I can't tell what you want. ..."


## 7. Browse disagreements

Pick two runs and look at the messages where they differ. This is the most useful thing to do with any eval:
the disagreements tell you whether the labels, the prompt or the model is at fault.

In [12]:
a = pd.DataFrame(tp.scored_rows("fresh/jev")).set_index("id")
b = pd.DataFrame(tp.scored_rows("fresh/native_sonnet")).set_index("id")
both = a.join(b[["pick", "outcome", "reply"]], rsuffix="_sonnet", how="inner")
diff = both[both["pick"].fillna("-") != both["pick_sonnet"].fillna("-")]
print(len(diff), "disagreements")
diff[["intent", "text", "pick", "outcome", "pick_sonnet", "outcome_sonnet"]].head(12)

114 disagreements


,intent,text,pick,outcome,pick_sonnet,outcome_sonnet
id,,,,,,
F0001,recover_password,where to retreive my key,hand_off_to_human,right,shopify_storefront.search_shop_policies_and_faqs,defensible
F0010,get_invoice,wanna get the bil from Chloe will uhelp me,hand_off_to_human,handed_off,NaN,asked
F0012,contact_customer_service,can yahelp me contacting customer support,hand_off_to_human,right,shopify_storefront.search_shop_policies_and_faqs,right
F0015,track_refund,would it be possible to check my compensation status,hand_off_to_human,handed_off,shopify_customer_account.get_most_recent_order_status,right
F0021,place_order,is it possible to shop an item,shopify_ucp.search_catalog,right,NaN,right
F0019,place_order,shopping article,shopify_ucp.search_catalog,right,NaN,right
F0029,payment_issue,i have a question about thepayments,hand_off_to_human,right,NaN,right
F0039,switch_account,assistance using the standard profile,hand_off_to_human,right,NaN,asked
F0053,registration_problems,question about opening user profiles,hand_off_to_human,right,NaN,asked


## 8. Make your own calls (optional)

With `OPENROUTER_API_KEY` and/or `TYPESAFE_API_KEY` in `.env`, the scripts in `scripts/` run any arm on any
number of messages and append to `runs/`. A few messages cost cents:

```bash
uv run python scripts/run_router.py --arm jev --sample fresh --limit 5
uv run python scripts/run_native.py --arm haiku --sample fresh --limit 5
```

The cell below shows one Jev call end to end.

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
if os.environ.get("TYPESAFE_API_KEY"):
    from toolpick.jev import Jev
    from toolpick.prompts import jev_instructions

    criteria = {t["name"]: t["description"] for t in tp.tools(customer_words=True)}
    out = Jev().choose("i wqant assistance checking purchase ORD-475206 status", jev_instructions(policy=True), criteria)
    a = out["answers"]["tool"]
    print(a["choice"], f"confidence {a['confidence']:.2f}", f"{out['latency_s']:.2f}s", f"${out['cost_usd']:.5f}")
    print(sorted(a["probabilities"].items(), key=lambda kv: -kv[1])[:4])
else:
    print("no TYPESAFE_API_KEY: skipping the live call")

no TYPESAFE_API_KEY: skipping the live call


## What to take away

1. Write the routing policy down and give it to every model; then you're measuring the model, not its guess at your policy.
2. Describe tools in your customers' words, not the vendor's.
3. Audit the answer key: check every failure by hand. Ours had nine mistakes.
4. Split wrong answers into benign and harmful. A hand-off and a bluff are not the same miss.
5. Compare on the same messages, with a paired test, and test the final setup on messages you didn't develop on.